# Tilt Census

Counts and distributions of tilt distance and direction by eddy type and region.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import seacofs_tilt_tools as tilt

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df_eddies, df_tilt = tilt.load_tilt_tables(paths, add_regions=True, grid=grid)

region_group_map = {"S1": "Shelf", "S2": "Shelf", "U1": "Upstream", "U2": "Upstream", "D1": "Downstream", "D2": "Downstream"}
df_eddies["RegionGroup"] = df_eddies.Region.map(region_group_map)
df_eddies.head()


In [ ]:
census = (
    df_eddies
    .groupby(["Cyc", "RegionGroup"], dropna=False)
    .agg(eddy_days=("Eddy", "size"), unique_eddies=("Eddy", "nunique"), tilt_days=("TiltDis", "count"), median_tilt=("TiltDis", "median"), mean_dir=("TiltDir", tilt.circular_mean_deg_true_north))
    .reset_index()
)
census


In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(9, 8), constrained_layout=True)
metrics = [("TiltDis", "Tilt distance (km)"), ("Age", "Age (days)"), ("Rc", "Radius (km)")]
for ax, (col, xlabel) in zip(axs, metrics):
    ae = df_eddies.loc[df_eddies.Cyc == "AE", col].dropna()
    ce = df_eddies.loc[df_eddies.Cyc == "CE", col].dropna()
    bins = tilt.shared_bins(ae, ce)
    tilt.mirrored_hist(ax, ae, ce, bins, xlabel)
axs[0].legend(frameon=False)
plt.show()


In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True, constrained_layout=True)
for ax, region in zip(axs, ["Shelf", "Upstream", "Downstream"]):
    subset = df_eddies.loc[df_eddies.RegionGroup == region]
    ae = subset.loc[subset.Cyc == "AE", "TiltDis"].dropna()
    ce = subset.loc[subset.Cyc == "CE", "TiltDis"].dropna()
    bins = tilt.shared_bins(ae, ce)
    tilt.mirrored_hist(ax, ae, ce, bins, "Tilt distance (km)")
    ax.set_title(region)
axs[0].legend(frameon=False)
plt.show()
